In [1]:
from neo4j_analysis import Neo4jAnalysis
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
import torch
import os

load_dotenv()
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USER")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
AGENT_RETRIEVAL_K = int(os.getenv("AGENT_RETRIEVAL_K", 10))

In [2]:
analysis = Neo4jAnalysis(NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD, NEO4J_DATABASE)

# Legal-domain model from Hugging Face, loaded via sentence-transformers.
MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
BATCH_SIZE = 128
FETCH_LIMIT = 5000
MAX_LENGTH = 512

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model = SentenceTransformer(MODEL_NAME, device=device)
model.max_seq_length = MAX_LENGTH

def embed_texts(texts):
    vectors = model.encode(
        texts,
        batch_size=BATCH_SIZE,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True,
    )
    return vectors.tolist()

fetch_query = """
MATCH (n)
WHERE n.text_embedding IS NULL
  AND (n.text IS NOT NULL OR n.title IS NOT NULL OR n.description IS NOT NULL)
WITH n,
     trim(
       coalesce(n.title + '\\n', '') +
       coalesce(n.description + '\\n', '') +
       coalesce(n.text, '')
     ) AS combinedText
WHERE combinedText <> ''
RETURN elementId(n) AS node_id, combinedText AS text
LIMIT $limit
"""

update_query = """
UNWIND $rows AS row
MATCH (n)
WHERE elementId(n) = row.node_id
SET n.text_embedding = row.embedding
RETURN count(n) AS updated
"""

total_updated = 0
while True:
    rows = analysis.run_query(fetch_query, {"limit": FETCH_LIMIT})
    if not rows:
        break

    texts = [r["text"] for r in rows]
    vectors = embed_texts(texts)

    payload = [
        {"node_id": r["node_id"], "embedding": v}
        for r, v in zip(rows, vectors)
    ]

    result = analysis.run_query(update_query, {"rows": payload})
    updated_now = result[0]["updated"] if result else 0
    total_updated += updated_now
    print(f"Updated {updated_now} nodes (total: {total_updated})")

print(f"Done. Total embedded nodes: {total_updated}")

No sentence-transformers model found with name nlpaueb/legal-bert-base-uncased. Creating a new one with mean pooling.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The p

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Updated 5000 nodes (total: 5000)


...

Updated 5000 nodes (total: 900000)


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Updated 5000 nodes (total: 905000)


Batches:   0%|          | 0/28 [00:00<?, ?it/s]

Updated 3579 nodes (total: 908579)
Done. Total embedded nodes: 908579


In [3]:
query = """
MATCH (n)
WHERE n.text_embedding IS NOT NULL
SET n:Text
"""

result = analysis.run_query(query)

In [4]:
query = """
CREATE VECTOR INDEX `text_embeddings_index` IF NOT EXISTS
FOR (n:Text) ON (n.text_embedding)
OPTIONS {
  indexConfig: {
    `vector.dimensions`: 768,
    `vector.similarity_function`: 'cosine'
  }
}
"""

print("Creating vector index...")
analysis.run_query(query)
print("Vector index created successfully!")

query = """
CREATE TEXT INDEX `text_index` IF NOT EXISTS
FOR (n:Text) ON (n.text)
"""
print("Creating text index...")
analysis.run_query(query)
print("Text index created successfully!")

Creating vector index...
Vector index created successfully!
Creating text index...
Text index created successfully!


In [5]:
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain, Neo4jVector
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import Tool
from langchain_core.tools import create_retriever_tool
from langchain.agents import create_agent
import random
import json
import re

# 1. Initialize the Core Components
print("Connecting to DB and loading models...")
graph = Neo4jGraph(
    url=NEO4J_URI,
    username=NEO4J_USER,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE
)

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
    api_key=GOOGLE_API_KEY,
    include_thoughts=True
) if GOOGLE_API_KEY else ChatOpenAI(
    model="gpt-5",
    temperature=0,
    api_key=OPENAI_API_KEY
)

embeddings = HuggingFaceEmbeddings(
    model_name="nlpaueb/legal-bert-base-uncased",
    encode_kwargs={"normalize_embeddings": True}
)

# Custom Cypher prompt tuned to schema and query patterns in this project
cypher_prompt = PromptTemplate(
    input_variables=["schema", "question"],
    template="""You are an expert Neo4j Cypher generator for a UK legislation graph.
Generate ONLY a valid read-only Cypher query.

Graph schema:
{schema}

Rules you MUST follow:
1) Return ONLY Cypher. No markdown, no commentary.
2) Read-only queries only. Never use CREATE, MERGE, DELETE, SET, CALL dbms.*, or schema/index changes.
3) When searching for titles, themes or topics, prefer the semantic search tool.
4) Prefer exact property names above and valid relationship directions.
5) When user names an Act/title, match with case-insensitive containment.
6) When user references a known legislation.gov.uk id, filter by l.uri CONTAINS 'ukpga/2010/4' style.
7) For network/visualization requests, return a path variable `p` (e.g., MATCH p=... RETURN p).
8) For tabular requests, RETURN explicit aliased columns and use ORDER BY/LIMIT when reasonable.
9) Avoid Cartesian products; always connect patterns.
10) Use OPTIONAL MATCH only when truly optional.
11) Keep traversal bounded for path exploration (e.g., *1..6 or *1..10).
12) CONTEXT IS MANDATORY for structural/text nodes. Include parent context up to Legislation.
13) Do not return a bare content node alone unless explicitly requested.
14) For relationship alternation, use ONE leading colon only, e.g. [:HAS_PART|HAS_CHAPTER|HAS_SECTION|HAS_SCHEDULE*0..3]. Never write [:HAS_PART|:HAS_CHAPTER|...].

Question: {question}"""
)

# Text2Cypher chain (kept as one tool among several granular tools)
cypher_chain = GraphCypherQAChain.from_llm(
    graph=graph,
    llm=llm,
    cypher_prompt=cypher_prompt,
    verbose=True,
    allow_dangerous_requests=True
)

# The Vector Search Tool
vector_store = Neo4jVector.from_existing_index(
    embedding=embeddings,
    url=NEO4J_URI,
    username=NEO4J_USER,
    password=NEO4J_PASSWORD,
    index_name="text_embeddings_index",
    node_label="Text",
    text_node_properties=["title", "description", "text"],
    embedding_node_property="text_embedding"
)

retriever = vector_store.as_retriever(search_kwargs={"k": AGENT_RETRIEVAL_K})
semantic_tool = create_retriever_tool(
    retriever,
    name="Semantic_Search",
    description="Use this for topic/meaning based retrieval over embedded text.",
)

VECTOR_INDEX_NAME = "text_embeddings_index"

def _parse_payload(payload):
    if not payload or not payload.strip():
        return {}
    try:
        return json.loads(payload)
    except Exception:
        return {"q": payload}

def _vector_hits(query_text: str, k: int = AGENT_RETRIEVAL_K):
    if not query_text or not query_text.strip():
        return []
    embedding = embeddings.embed_query(query_text)
    query = """
    CALL db.index.vector.queryNodes($index_name, $k, $embedding)
    YIELD node, score
    RETURN elementId(node) AS node_id,
           labels(node) AS labels,
           score,
           coalesce(node.title, node.text, node.description) AS matched_content,
           node.title AS node_title,
           node.uri AS node_uri
    ORDER BY score DESC
    """
    return analysis.run_query(
        query,
        {
            "index_name": VECTOR_INDEX_NAME,
            "k": k,
            "embedding": embedding,
        },
    )

def schema_navigation(_: str = "") -> str:
    node_query = """
    CALL apoc.meta.data()
    YIELD label, property, type, elementType
    WHERE elementType = "node" 
      AND type <> "RELATIONSHIP"
      AND label <> "Text"
    RETURN label, collect(property + ': ' + type) AS properties
    """
    nodes = analysis.run_query(node_query)

    rel_query = """
    MATCH (a)-[r]->(b)
    WITH [l IN labels(a) WHERE l <> 'Text'] AS start_labels, 
        type(r) AS relationship_type, 
        [l IN labels(b) WHERE l <> 'Text'] AS end_labels
    WHERE size(start_labels) > 0 AND size(end_labels) > 0
    UNWIND start_labels AS start_label
    UNWIND end_labels AS end_label
    RETURN DISTINCT start_label, relationship_type, end_label
    LIMIT 5000
    """
    rels = analysis.run_query(rel_query)

    schema_text = "GRAPH SCHEMA DEFINITION:\n\n"
    schema_text += "Node Labels and Properties:\n"
    for node in nodes:
        props = ", ".join(node["properties"]) if node["properties"] else "No properties"
        schema_text += f"   - (:{node['label']} {{ {props} }})\n"

    schema_text += "\nValid Relationship Connections:\n"
    if rels:
        for rel in rels:
            schema_text += f"   - (:{rel['start_label']})-[:{rel['relationship_type']}]->(:{rel['end_label']})\n"
    else:
        schema_text += "   - No relationships found.\n"

    return schema_text

def find_legislation(payload: str):
    data = _parse_payload(payload)
    q = data.get("q", "")
    k = int(data.get("k", AGENT_RETRIEVAL_K))
    hits = _vector_hits(q, k=k)
    if not hits:
        return []

    query = """
    UNWIND $hits AS h
    MATCH (hit) WHERE elementId(hit) = h.node_id
    OPTIONAL MATCH (l_direct:Legislation) WHERE elementId(l_direct) = h.node_id
    OPTIONAL MATCH (l_ctx:Legislation)-[:HAS_PART|HAS_CHAPTER|HAS_SECTION|HAS_PARAGRAPH|HAS_SCHEDULE|HAS_SUBPARAGRAPH|HAS_EXPLANATORY_NOTES*1..6]->(hit)
    WITH h, coalesce(l_direct, l_ctx) AS l
    WHERE l IS NOT NULL
    RETURN DISTINCT l.title AS title,
                    l.uri AS uri,
                    l.enactment_date AS enactment_date,
                    l.status AS status,
                    l.category AS category,
                    h.score AS vector_score
    ORDER BY vector_score DESC, enactment_date DESC
    LIMIT 25
    """
    return analysis.run_query(query, {"hits": hits})

def get_legislation_context(payload: str):
    data = _parse_payload(payload)
    q = data.get("q", "")
    k = int(data.get("k", AGENT_RETRIEVAL_K))
    hits = _vector_hits(q, k=k)
    if not hits:
        return []

    query = """
    UNWIND $hits AS h
    MATCH (hit) WHERE elementId(hit) = h.node_id
    OPTIONAL MATCH (l_direct:Legislation) WHERE elementId(l_direct) = h.node_id
    OPTIONAL MATCH (l_ctx:Legislation)-[:HAS_PART|HAS_CHAPTER|HAS_SECTION|HAS_PARAGRAPH|HAS_SCHEDULE|HAS_SUBPARAGRAPH|HAS_EXPLANATORY_NOTES*1..6]->(hit)
    WITH h, coalesce(l_direct, l_ctx) AS l
    WHERE l IS NOT NULL
    OPTIONAL MATCH (l)-[:HAS_PART]->(part:Part)
    OPTIONAL MATCH (part)-[:HAS_CHAPTER]->(chapter:Chapter)
    OPTIONAL MATCH (chapter)-[:HAS_SECTION]->(section:Section)
    OPTIONAL MATCH (section)-[:HAS_PARAGRAPH]->(paragraph:Paragraph)
    RETURN l.title AS legislation_title,
           l.uri AS legislation_uri,
           count(DISTINCT part) AS part_count,
           count(DISTINCT chapter) AS chapter_count,
           count(DISTINCT section) AS section_count,
           count(DISTINCT paragraph) AS paragraph_count,
           max(h.score) AS vector_score
    ORDER BY vector_score DESC, paragraph_count DESC
    LIMIT 5
    """
    return analysis.run_query(query, {"hits": hits})

def retrieve_text_with_context(payload: str):
    data = _parse_payload(payload)
    q = data.get("q", "")
    k = int(data.get("k", AGENT_RETRIEVAL_K))
    limit = int(data.get("limit", 15))
    hits = _vector_hits(q, k=k)
    if not hits:
        return []

    query = """
    UNWIND $hits AS h
    MATCH (n) WHERE elementId(n) = h.node_id
    OPTIONAL MATCH p=(l:Legislation)-[:HAS_PART|HAS_CHAPTER|HAS_SECTION|HAS_PARAGRAPH|HAS_SCHEDULE|HAS_SUBPARAGRAPH|HAS_EXPLANATORY_NOTES*0..6]->(n)
    WITH h, n, l, p,
         head([x IN nodes(p) WHERE x:Part]) AS part,
         head([x IN nodes(p) WHERE x:Chapter]) AS chapter,
         head([x IN nodes(p) WHERE x:Section]) AS section,
         head([x IN nodes(p) WHERE x:Paragraph]) AS paragraph
    WHERE l IS NOT NULL
    RETURN DISTINCT l.title AS legislation_title,
           l.uri AS legislation_uri,
           part.number AS part_number,
           part.title AS part_title,
           part.restrict_start_date AS part_restrict_start_date,
           part.restrict_end_date AS part_restrict_end_date,
           part.restrict_extent AS part_restrict_extent,
           part.status AS part_status,
           chapter.number AS chapter_number,
           chapter.title AS chapter_title,
           chapter.restrict_start_date AS chapter_restrict_start_date,
           chapter.restrict_end_date AS chapter_restrict_end_date,
           chapter.restrict_extent AS chapter_restrict_extent,
           chapter.status AS chapter_status,
           section.number AS section_number,
           section.title AS section_title,
           section.restrict_start_date AS section_restrict_start_date,
           section.restrict_end_date AS section_restrict_end_date,
           section.restrict_extent AS section_restrict_extent,
           section.status AS section_status,
           paragraph.number AS paragraph_number,
           paragraph.restrict_start_date AS paragraph_restrict_start_date,
           paragraph.restrict_end_date AS paragraph_restrict_end_date,
           paragraph.restrict_extent AS paragraph_restrict_extent,
           paragraph.status AS paragraph_status,
           coalesce(paragraph.text, n.text, n.title, n.description) AS matched_text,
           h.score AS vector_score
    ORDER BY vector_score DESC
    LIMIT $limit
    """
    return analysis.run_query(query, {"hits": hits, "limit": limit})

def citation_reasoning(payload: str):
    data = _parse_payload(payload)
    q = data.get("q", "")
    k = int(data.get("k", AGENT_RETRIEVAL_K))
    hits = _vector_hits(q, k=k)
    if not hits:
        return []

    query = """
    UNWIND $hits AS h
    MATCH (hit) WHERE elementId(hit) = h.node_id
    OPTIONAL MATCH (source_direct:Legislation) WHERE elementId(source_direct) = h.node_id
    OPTIONAL MATCH (source_ctx:Legislation)-[:HAS_PART|HAS_CHAPTER|HAS_SECTION|HAS_PARAGRAPH|HAS_SCHEDULE|HAS_SUBPARAGRAPH|HAS_EXPLANATORY_NOTES*1..6]->(hit)
    WITH h, coalesce(source_direct, source_ctx) AS source
    WHERE source IS NOT NULL
    OPTIONAL MATCH (source)-[r:CITES]->(target:Legislation)
    RETURN source.title AS source_title,
           source.uri AS source_uri,
           target.title AS target_title,
           target.uri AS target_uri,
           type(r) AS relationship_type,
           h.score AS vector_score
    ORDER BY vector_score DESC
    LIMIT 20
    """
    return analysis.run_query(query, {"hits": hits})

def supersedes_chain(payload: str):
    data = _parse_payload(payload)
    q = data.get("q", "")
    query = """
    MATCH (source:Legislation)
    WHERE toLower(coalesce(source.title, "")) CONTAINS toLower($q)
       OR toLower(coalesce(source.uri, "")) CONTAINS toLower($q)
    OPTIONAL MATCH (source)-[:SUPERSEDES]->(target:Legislation)
    RETURN source.title AS source_title,
           source.uri AS source_uri,
           target.title AS target_title,
           target.uri AS target_uri
    LIMIT 20
    """
    return analysis.run_query(query, {"q": q})

def superseded_chain(payload: str):
    data = _parse_payload(payload)
    q = data.get("q", "")
    query = """
    MATCH (source:Legislation)
    WHERE toLower(coalesce(source.title, "")) CONTAINS toLower($q)
       OR toLower(coalesce(source.uri, "")) CONTAINS toLower($q)
    OPTIONAL MATCH (source)-[:SUPERSEDED_BY]->(target:Legislation)
    RETURN source.title AS source_title,
           source.uri AS source_uri,
           target.title AS target_title,
           target.uri AS target_uri
    LIMIT 20
    """
    return analysis.run_query(query, {"q": q})

def read_only_cypher(payload: str):
    forbidden = r"\b(CREATE|MERGE|DELETE|DETACH|SET|DROP|REMOVE)\b"
    if re.search(forbidden, payload, flags=re.IGNORECASE):
        return {"error": "Only read-only Cypher is allowed in this tool."}
    # Normalize a common Neo4j syntax mistake from generated queries:
    # [:A|:B|:C]  ->  [:A|B|C]
    normalized_payload = payload.replace('|:', '|')
    return analysis.run_query(normalized_payload)

graph_schema_tool = Tool(
    name="Graph_Schema_Navigator",
    func=schema_navigation,
    description="Get graph labels, properties and relationship types. Input can be empty.",
)

legislation_finder_tool = Tool(
    name="Legislation_Finder",
    func=find_legislation,
    description="Find legislation via vector index hits mapped to parent Legislation. Input: JSON string like {\"q\":\"corporation tax\",\"k\":20}.",
)

context_tool = Tool(
    name="Legislation_Context_Summary",
    func=get_legislation_context,
    description="Get legislation hierarchy summary using vector-index seeds. Input: JSON string like {\"q\":\"ukpga/2010/4\",\"k\":20}.",
)

text_context_tool = Tool(
    name="Contextual_Text_Retriever",
    func=retrieve_text_with_context,
    description="Retrieve context-rich text using vector index (with legislation/part/chapter/section/paragraph context, including restrict_start_date, restrict_end_date, restrict_extent and status). Input: JSON string like {\"q\":\"corporate tax\",\"k\":20,\"limit\":10}.",
)

citation_tool = Tool(
    name="Citation_Network_Explorer",
    func=citation_reasoning,
    description="Use vector-index seeds to find relevant legislation, then expand outgoing citation links.",
)

supersedes_tool = Tool(
    name="Supersedes_Network_Explorer",
    func=supersedes_chain,
    description="Get outgoing supersedes network for a legislation using SUPERSEDES. Input: JSON string like {\"q\":\"Value Added Tax Act\"}.",
)

superseded_tool = Tool(
    name="Superseded_By_Network_Explorer",
    func=superseded_chain,
    description="Get incoming superseded_by network for a legislation using SUPERSEDED_BY. Input: JSON string like {\"q\":\"Value Added Tax Act\"}.",
)

safe_cypher_tool = Tool(
    name="Read_Only_Cypher",
    func=read_only_cypher,
    description="Execute ad-hoc read-only Cypher. Input: Cypher query string.",
)

text2cypher_tool = Tool(
    name="Text2Cypher_Expert",
    func=cypher_chain.invoke,
    description="Translate complex natural language questions into Cypher when other tools are insufficient.",
)

# Compile the agent with granular capabilities
tools = [
    graph_schema_tool,
    legislation_finder_tool,
    context_tool,
    text_context_tool,
    citation_tool,
    supersedes_tool,
    superseded_tool,
    safe_cypher_tool,
    text2cypher_tool,
    semantic_tool,
]

system_prompt = """You are a highly capable legal AI assistant.
Use the most specific tool first. Use the Graph_Schema_Navigator before other tools to understand the schema. Prefer granular tools before Text2Cypher_Expert.
For retrieval tasks, prefer vector-index-backed tools (Legislation_Finder, Legislation_Context_Summary, Contextual_Text_Retriever, Citation_Network_Explorer, Semantic_Search).
Always preserve legal context (Legislation > Part > Chapter > Section > Paragraph) when answering content questions.
If a tool returns empty results, do not repeat the exact same call. Always include links to relevant legislation, sections and parts in your responses."""

agent_executor = create_agent(llm, tools, system_prompt=system_prompt)

# Test
user_questions = []
user_questions.append("Find legislation that discusses the nuances of Corporate Tax.")
#user_questions.append("Which legislation superseded the Help–to–Save Accounts Regulations?")
#user_questions.append("What is the small profits rate for corporations?")
#user_questions.append("What are the rules for reimbursements of overpaid corporation tax?")
#user_questions.append("How many citations does the Value Added Tax Act have?")
#user_questions.append("How has the Value Added Tax Act changed over time?")

Connecting to DB and loading models...


No sentence-transformers model found with name nlpaueb/legal-bert-base-uncased. Creating a new one with mean pooling.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
# Phase 1 tools: deterministic lookup, hierarchy resolution, citation metrics, grounded answers
def legislation_by_uri(payload: str):
    data = _parse_payload(payload)
    uri = data.get("uri") or data.get("q", "")
    if not uri:
        return {"error": "Provide 'uri' (or 'q') in payload."}

    query = """
    MATCH (l:Legislation)
    WHERE l.uri = $uri OR l.uri CONTAINS $uri
    RETURN l.title AS title,
           l.uri AS uri,
           l.enactment_date AS enactment_date,
           l.status AS status,
           l.category AS category
    ORDER BY l.enactment_date DESC
    LIMIT 5
    """
    return analysis.run_query(query, {"uri": uri})

def hierarchy_path_resolver(payload: str):
    data = _parse_payload(payload)
    node_id = data.get("node_id")
    uri = data.get("uri")

    if not node_id and not uri:
        return {"error": "Provide 'node_id' (elementId) or 'uri'."}

    query_by_node = """
    MATCH (n)
    WHERE elementId(n) = $node_id
    OPTIONAL MATCH p=(l:Legislation)-[:HAS_PART|HAS_CHAPTER|HAS_SECTION|HAS_PARAGRAPH|HAS_SCHEDULE|HAS_SUBPARAGRAPH|HAS_EXPLANATORY_NOTES*0..6]->(n)
    WITH n, l, p,
         head([x IN nodes(p) WHERE x:Part]) AS part,
         head([x IN nodes(p) WHERE x:Chapter]) AS chapter,
         head([x IN nodes(p) WHERE x:Section]) AS section,
         head([x IN nodes(p) WHERE x:Paragraph]) AS paragraph
    RETURN labels(n) AS node_labels,
           coalesce(n.uri, n.id, elementId(n)) AS node_ref,
           l.title AS legislation_title,
           l.uri AS legislation_uri,
           part.number AS part_number,
           part.title AS part_title,
           part.restrict_start_date AS part_restrict_start_date,
           part.restrict_end_date AS part_restrict_end_date,
           part.restrict_extent AS part_restrict_extent,
           part.status AS part_status,
           chapter.number AS chapter_number,
           chapter.title AS chapter_title,
           chapter.restrict_start_date AS chapter_restrict_start_date,
           chapter.restrict_end_date AS chapter_restrict_end_date,
           chapter.restrict_extent AS chapter_restrict_extent,
           chapter.status AS chapter_status,
           section.number AS section_number,
           section.title AS section_title,
           section.restrict_start_date AS section_restrict_start_date,
           section.restrict_end_date AS section_restrict_end_date,
           section.restrict_extent AS section_restrict_extent,
           section.status AS section_status,
           paragraph.number AS paragraph_number,
           paragraph.restrict_start_date AS paragraph_restrict_start_date,
           paragraph.restrict_end_date AS paragraph_restrict_end_date,
           paragraph.restrict_extent AS paragraph_restrict_extent,
           paragraph.status AS paragraph_status
    LIMIT 10
    """

    query_by_uri = """
    MATCH (n)
    WHERE n.uri = $uri OR n.uri CONTAINS $uri
    OPTIONAL MATCH p=(l:Legislation)-[:HAS_PART|HAS_CHAPTER|HAS_SECTION|HAS_PARAGRAPH|HAS_SCHEDULE|HAS_SUBPARAGRAPH|HAS_EXPLANATORY_NOTES*0..6]->(n)
    WITH n, l, p,
         head([x IN nodes(p) WHERE x:Part]) AS part,
         head([x IN nodes(p) WHERE x:Chapter]) AS chapter,
         head([x IN nodes(p) WHERE x:Section]) AS section,
         head([x IN nodes(p) WHERE x:Paragraph]) AS paragraph
    RETURN labels(n) AS node_labels,
           coalesce(n.uri, n.id, elementId(n)) AS node_ref,
           l.title AS legislation_title,
           l.uri AS legislation_uri,
           part.number AS part_number,
           part.title AS part_title,
           part.restrict_start_date AS part_restrict_start_date,
           part.restrict_end_date AS part_restrict_end_date,
           part.restrict_extent AS part_restrict_extent,
           part.status AS part_status,
           chapter.number AS chapter_number,
           chapter.title AS chapter_title,
           chapter.restrict_start_date AS chapter_restrict_start_date,
           chapter.restrict_end_date AS chapter_restrict_end_date,
           chapter.restrict_extent AS chapter_restrict_extent,
           chapter.status AS chapter_status,
           section.number AS section_number,
           section.title AS section_title,
           section.restrict_start_date AS section_restrict_start_date,
           section.restrict_end_date AS section_restrict_end_date,
           section.restrict_extent AS section_restrict_extent,
           section.status AS section_status,
           paragraph.number AS paragraph_number,
           paragraph.restrict_start_date AS paragraph_restrict_start_date,
           paragraph.restrict_end_date AS paragraph_restrict_end_date,
           paragraph.restrict_extent AS paragraph_restrict_extent,
           paragraph.status AS paragraph_status
    LIMIT 10
    """

    if node_id:
        return analysis.run_query(query_by_node, {"node_id": node_id})
    return analysis.run_query(query_by_uri, {"uri": uri})

def citation_counts(payload: str):
    data = _parse_payload(payload)
    q = data.get("uri") or data.get("q", "")
    if not q:
        return {"error": "Provide 'uri' or 'q'."}

    query = """
    MATCH (l:Legislation)
    WHERE l.uri = $q OR l.uri CONTAINS $q OR toLower(coalesce(l.title, "")) CONTAINS toLower($q)
    CALL(l) {
      WITH l
      OPTIONAL MATCH (l)-[:LINKED_TO]->(t:Legislation)
      RETURN count(DISTINCT t) AS outgoing_count, collect(DISTINCT t.title)[0..5] AS top_outgoing_titles
    }
    CALL(l) {
      WITH l
      OPTIONAL MATCH (s:Legislation)-[:LINKED_TO]->(l)
      RETURN count(DISTINCT s) AS incoming_count, collect(DISTINCT s.title)[0..5] AS top_incoming_titles
    }
    RETURN l.title AS legislation_title,
           l.uri AS legislation_uri,
           outgoing_count,
           incoming_count,
           top_outgoing_titles,
           top_incoming_titles
    LIMIT 5
    """
    return analysis.run_query(query, {"q": q})

def answer_grounder(payload: str):
    data = _parse_payload(payload)
    answer = data.get("answer", "")
    evidence = data.get("evidence", [])
    question = data.get("q", "")

    if not evidence and question:
        hits = _vector_hits(question, k=min(AGENT_RETRIEVAL_K, 8))
        evidence = [
            {
                "node_id": h.get("node_id"),
                "uri": h.get("node_uri"),
                "title": h.get("node_title"),
                "score": h.get("score"),
            }
            for h in hits
        ]

    normalized = []
    seen = set()
    for e in evidence if isinstance(evidence, list) else []:
        if not isinstance(e, dict):
            continue
        key = (e.get("uri"), e.get("node_id"), e.get("title"))
        if key in seen:
            continue
        seen.add(key)
        normalized.append(
            {
                "uri": e.get("uri"),
                "title": e.get("title"),
                "node_id": e.get("node_id"),
                "score": e.get("score"),
            }
        )

    return {
        "grounded_answer": answer,
        "citation_count": len(normalized),
        "citations": normalized,
    }

legislation_by_uri_tool = Tool(
    name="Legislation_By_URI",
    func=legislation_by_uri,
    description="Deterministic legislation lookup by URI. Input JSON: {\"uri\":\"http://www.legislation.gov.uk/...\"}.",
)

hierarchy_path_resolver_tool = Tool(
    name="Hierarchy_Path_Resolver",
    func=hierarchy_path_resolver,
    description="Resolve full hierarchy context for a node by elementId or uri, including restrict_start_date, restrict_end_date, restrict_extent and status for Part/Chapter/Section/Paragraph. Input JSON: {\"node_id\":\"...\"} or {\"uri\":\"...\"}.",
)

citation_counts_tool = Tool(
    name="Citation_Counts",
    func=citation_counts,
    description="Get inbound/outbound citation counts and top linked Acts. Input JSON: {\"q\":\"Value Added Tax Act\"} or {\"uri\":\"...\"}.",
)

answer_grounder_tool = Tool(
    name="Answer_Grounder",
    func=answer_grounder,
    description="Return a grounded answer object with evidence/citations. Input JSON: {\"answer\":\"...\",\"evidence\":[...]} or {\"q\":\"...\"}.",
)

# Merge Phase 1 tools into current toolset without duplicates
_phase1 = [
    legislation_by_uri_tool,
    hierarchy_path_resolver_tool,
    citation_counts_tool,
    answer_grounder_tool,
]
_tool_map = {t.name: t for t in tools}
for t in _phase1:
    _tool_map[t.name] = t
tools = list(_tool_map.values())

system_prompt_phase1 = system_prompt + "\nUse Legislation_By_URI for exact act lookup, Hierarchy_Path_Resolver for context reconstruction, Citation_Counts for quick citation metrics, and Answer_Grounder to ground final responses."
agent_executor = create_agent(llm, tools, system_prompt=system_prompt_phase1)
print("Phase 1 tools added and agent rebuilt.")

Phase 1 tools added and agent rebuilt.


In [7]:
# Streaming execution with pretty JSON-formatted reasoning + tool timings
import time
from collections import defaultdict, deque

user_question = random.choice(user_questions)
print(f"\nUser asked: '{user_question}'\n")

def _pretty(obj):
    print(json.dumps(obj, indent=2, ensure_ascii=False, default=str))

def _pretty_json_content(value):
    # Try to pretty-print JSON strings/dicts/lists in tool outputs
    if isinstance(value, (dict, list)):
        return value

    if isinstance(value, str):
        text = value.strip()
        if not text:
            return ""
        try:
            return json.loads(text)
        except Exception:
            # Some tool outputs include prose plus JSON; attempt a best-effort extraction
            start_obj = text.find("{")
            end_obj = text.rfind("}")
            if start_obj != -1 and end_obj != -1 and end_obj > start_obj:
                candidate = text[start_obj : end_obj + 1]
                try:
                    return json.loads(candidate)
                except Exception:
                    pass
            start_arr = text.find("[")
            end_arr = text.rfind("]")
            if start_arr != -1 and end_arr != -1 and end_arr > start_arr:
                candidate = text[start_arr : end_arr + 1]
                try:
                    return json.loads(candidate)
                except Exception:
                    pass
            return value

    return str(value)

final_answer = ""
event_idx = 0
run_start = time.perf_counter()
tool_start_times = defaultdict(deque)  # tool_name -> queue of start times
print("--- Live Agent Stream (JSON) ---\n")

for event in agent_executor.stream(
    {"messages": [("user", user_question)]},
    stream_mode="updates",
):
    if not isinstance(event, dict):
        continue

    event_idx += 1
    elapsed = round(time.perf_counter() - run_start, 3)
    _pretty({"event": event_idx, "elapsed_s": elapsed, "keys": list(event.keys())})

    for node_name, data in event.items():
        messages = data.get("messages", []) if isinstance(data, dict) else []
        if not messages:
            continue

        msg = messages[-1]
        msg_type = getattr(msg, "type", None)

        if msg_type == "ai" and getattr(msg, "tool_calls", None):
            for tool_call in msg.tool_calls:
                tool_name = tool_call.get("name", "unknown")
                started_at = time.perf_counter()
                tool_start_times[tool_name].append(started_at)
                _pretty(
                    {
                        "event": event_idx,
                        "node": node_name,
                        "type": "tool_call",
                        "tool_name": tool_name,
                        "args": tool_call.get("args", {}),
                        "started_at_s": round(started_at - run_start, 3),
                    }
                )

        elif msg_type == "tool":
            raw_content = getattr(msg, "content", "")
            pretty_content = _pretty_json_content(raw_content)
            tool_name = getattr(msg, "name", "unknown")
            finished_at = time.perf_counter()

            duration_s = None
            if tool_start_times[tool_name]:
                started_at = tool_start_times[tool_name].popleft()
                duration_s = round(finished_at - started_at, 3)

            _pretty(
                {
                    "event": event_idx,
                    "node": node_name,
                    "type": "tool_result",
                    "tool_name": tool_name,
                    "duration_s": duration_s,
                    "finished_at_s": round(finished_at - run_start, 3),
                    "content_preview": pretty_content,
                }
            )

        elif msg_type == "ai":
            content = getattr(msg, "content", "")
            if isinstance(content, list):
                text_blocks = [b.get("text", "") for b in content if isinstance(b, dict)]
                text = "\n".join([t for t in text_blocks if t]).strip()
            else:
                text = str(content).strip()

            if text:
                final_answer = text
                _pretty(
                    {
                        "event": event_idx,
                        "node": node_name,
                        "type": "ai_response",
                        "text": text,
                    }
                )

print("\n--- Final Answer ---")
print(final_answer)
print(f"\nTotal elapsed: {round(time.perf_counter() - run_start, 3)}s")


User asked: 'Find legislation that discusses the nuances of Corporate Tax.'

--- Live Agent Stream (JSON) ---

{
  "event": 1,
  "elapsed_s": 55.929,
  "keys": [
    "model"
  ]
}
{
  "event": 1,
  "node": "model",
  "type": "tool_call",
  "tool_name": "Graph_Schema_Navigator",
  "args": {
    "__arg1": ""
  },
  "started_at_s": 55.93
}
{
  "event": 2,
  "elapsed_s": 61.76,
  "keys": [
    "tools"
  ]
}
{
  "event": 2,
  "node": "tools",
  "type": "tool_result",
  "tool_name": "Graph_Schema_Navigator",
  "duration_s": 5.83,
  "finished_at_s": 61.76,
  "content_preview": "GRAPH SCHEMA DEFINITION:\n\nNode Labels and Properties:\n   - (:Legislation { uri: STRING, title: STRING, description: STRING, modified_date: DATE, valid_date: DATE, enactment_date: DATE, status: STRING, category: STRING, text_embedding: LIST, coming_into_force: DATE })\n   - (:Part { uri: STRING, id: STRING, title: STRING, number: STRING, order: INTEGER, text_embedding: LIST, restrict_start_date: DATE, restrict_end_d